In [0]:
%sql

-- FACT TRIP TABLE --

CREATE OR REPLACE TABLE nyc_mobility.mart.fact_trip AS

-- Step 1: Pre-calculate date key and hour in memory (Gold Layer only)
WITH prep_green_taxi AS (
    SELECT 
        *,
        CAST(date_format(lpep_pickup_datetime, 'yyyyMMdd') AS INT) AS pickup_date_key,
        hour(lpep_pickup_datetime) AS pickup_hour
    FROM nyc_mobility.clean.green_taxi
    WHERE PULocationID IS NOT NULL
      AND DOLocationID IS NOT NULL
      AND lpep_pickup_datetime IS NOT NULL
)

-- Step 2: Build Fact Table with simple, direct JOINs
SELECT 
    -- Primary Key
    t.trip_id,
    
    -- Foreign Keys
    t.PULocationID AS pu_location_id,
    t.DOLocationID AS do_location_id,
    t.pickup_date_key,
    w.weather_id,
    a.advisory_id,
    
    -- Extracted Attributes
    t.lpep_pickup_datetime AS pickup_datetime,
    t.lpep_dropoff_datetime AS dropoff_datetime,
    t.pickup_hour,
    
    -- Measures
    t.passenger_count,
    t.trip_distance,
    t.trip_duration_min,
    t.fare_amount,
    t.total_amount

FROM prep_green_taxi t

-- 1. Join dim_zone (Pickup Location)
LEFT JOIN nyc_mobility.mart.dim_zone AS z 
    ON t.PULocationID = z.location_id

-- 2. Join dim_date directly on INT key
LEFT JOIN nyc_mobility.mart.dim_date AS d
    ON t.pickup_date_key = d.date_key

-- 3. Join dim_weather on INT key + Hour + Borough
LEFT JOIN nyc_mobility.mart.dim_weather AS w 
    ON t.pickup_date_key = w.date_key
        AND t.pickup_hour = w.hour_of_day
        AND z.borough = w.borough

-- 4. Join dim_advisory on Borough + Active Timestamp Window
LEFT JOIN nyc_mobility.mart.dim_advisory AS a 
    ON z.borough = a.borough
        AND t.lpep_pickup_datetime BETWEEN a.effective_from AND a.effective_to;